# MambaIR-Scratch V3: SIDL Scratch 복원
**개선사항 (V2 대비)**
- patch_size 64 → 128 (더 넓은 context 학습)
- w_edge 0.1 → 0.2 (스크래치 경계 선명도 강화)
- FFT Loss 추가 (주파수 도메인 손실)
- 로컬 디스크 캐시 (I/O 5~10배 향상)
- TTA 추론 (8방향 앙상블, +0.2~0.5dB)
- archs 중복 등록 버그 수정
- Resume 학습 지원

**실행 순서**: 셀을 위에서 아래로 순서대로 실행

## 1. Drive 마운트 & MambaIR 클론
Google Drive 연결 후 MambaIR 레포가 없으면 클론합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')

if not os.path.exists('/content/drive/MyDrive/MambaIR'):
    !git clone https://github.com/csguoh/MambaIR
    print('클론 완료')
else:
    print('이미 존재, 스킵')

os.chdir('/content/drive/MyDrive/MambaIR')
print('완료:', os.getcwd())

Mounted at /content/drive
이미 존재, 스킵
완료: /content/drive/MyDrive/MambaIR


## 2. 환경 설치 (처음 1회만)
mamba-ssm, causal-conv1d 설치 및 MambaIR setup. **약 20분 소요.**
런타임 재시작 후에는 셀 3(환경 패치)만 실행하면 됩니다.

In [2]:
import os
os.chdir('/content/drive/MyDrive/MambaIR')

!pip install mamba-ssm --no-build-isolation -q
!pip install causal-conv1d --no-build-isolation -q
!pip install einops timm -q

!python setup.py develop --no_cuda_ext 2>/dev/null || echo 'setup 완료'
print('환경 설치 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 22.4 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
setup 완료
환경 설치 완료


## 3. 환경 패치 (런타임 재시작 후 매번 실행)
**런타임 재시작할 때마다 반드시 실행해야 합니다.**
- torchvision 호환 패치
- pip basicsr 제거 → MambaIR 레포 버전 사용
- base_model CosineAnnealingLR 패치
- archs __init__.py 중복 등록 버그 수정

In [3]:
import site, os

# 1. torchvision 패치
for path in site.getsitepackages():
    target = os.path.join(path, 'basicsr/data/degradations.py')
    if os.path.exists(target):
        with open(target, 'r') as f:
            content = f.read()
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(target, 'w') as f:
            f.write(content)
        print('torchvision 패치 완료')
        break

# 2. pip basicsr 제거 -> MambaIR 레포 버전
!pip uninstall basicsr -y -q
os.chdir('/content/drive/MyDrive/MambaIR')
!python setup.py develop --no_cuda_ext 2>/dev/null || echo 'setup 완료'

# 3. base_model CosineAnnealingLR 패치
path = '/content/drive/MyDrive/MambaIR/basicsr/models/base_model.py'
with open(path, 'r') as f:
    content = f.read()
old = """        else:\n            if scheduler_type == 'CosineAnnealingLR':\n                schedulers.append(\n                    CosineAnnealingLR(\n                        self.optimizer_g,\n                        T_max=train_opt['scheduler']['T_max'],\n                        eta_min=train_opt['scheduler'].get('eta_min', 0)\n                    )\n                )\n            else:\n                raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""
new = """        elif scheduler_type == 'CosineAnnealingLR':\n            for optimizer in self.optimizers:\n                self.schedulers.append(\n                    torch.optim.lr_scheduler.CosineAnnealingLR(\n                        optimizer,\n                        T_max=train_opt['scheduler']['T_max'],\n                        eta_min=train_opt['scheduler'].get('eta_min', 0)\n                    )\n                )\n        else:\n            raise NotImplementedError(f'Scheduler {scheduler_type} is not implemented yet.')"""
if old in content:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('base_model CosineAnnealingLR 패치 완료')
else:
    print('base_model 패치 이미 적용됨')

# 4. archs __init__.py 중복 등록 버그 수정
arch_init = '/content/drive/MyDrive/MambaIR/basicsr/archs/__init__.py'
clean_init = '''import importlib
from copy import deepcopy
from os import path as osp

from basicsr.utils import get_root_logger, scandir
from basicsr.utils.registry import ARCH_REGISTRY

__all__ = ['build_network']

arch_folder = osp.dirname(osp.abspath(__file__))
arch_filenames = [osp.splitext(osp.basename(v))[0] for v in scandir(arch_folder) if v.endswith('_arch.py')]
_arch_modules = [importlib.import_module(f'basicsr.archs.{file_name}') for file_name in arch_filenames]


def build_network(opt):
    opt = deepcopy(opt)
    network_type = opt.pop('type')
    net = ARCH_REGISTRY.get(network_type)(**opt)
    logger = get_root_logger()
    logger.info(f'Network [{net.__class__.__name__}] is created.')
    return net
'''
with open(arch_init, 'w') as f:
    f.write(clean_init)
print('archs __init__.py 중복 등록 수정 완료')

print('\n모든 패치 완료!')

setup 완료
base_model 패치 이미 적용됨
archs __init__.py 중복 등록 수정 완료

모든 패치 완료!


## 4. 커스텀 파일 배치
Drive의 SIDL_custom 폴더에서 arch, loss 파일을 MambaIR 레포에 복사합니다.
파일이 없으면 Drive에 업로드 후 실행하세요.

In [4]:
import shutil, os

ARCH_SRC = '/content/drive/MyDrive/SIDL_custom/mambair_scratch_arch.py'
LOSS_SRC = '/content/drive/MyDrive/SIDL_custom/scratch_losses.py'
ARCH_DST = '/content/drive/MyDrive/MambaIR/basicsr/archs/mambair_scratch_arch.py'
LOSS_DST = '/content/drive/MyDrive/MambaIR/basicsr/losses/scratch_losses.py'

shutil.copy(ARCH_SRC, ARCH_DST)
shutil.copy(LOSS_SRC, LOSS_DST)
print('arch, loss 파일 복사 완료')

# losses __init__.py에 ScratchLoss import 추가
loss_init = '/content/drive/MyDrive/MambaIR/basicsr/losses/__init__.py'
with open(loss_init, 'r') as f:
    content = f.read()
line = 'from .scratch_losses import ScratchLoss'
if line not in content:
    with open(loss_init, 'a') as f:
        f.write('\n' + line + '\n')
    print('ScratchLoss import 추가됨')
else:
    print('ScratchLoss import 이미 존재')

arch, loss 파일 복사 완료
ScratchLoss import 이미 존재


## 5. 데이터 로컬 캐시
Drive에서 로컬 SSD로 데이터를 복사합니다. **I/O 속도가 5~10배 빨라집니다.**
`scratch_dataset.zip`이 Drive에 있어야 합니다. 이미 있으면 스킵됩니다.

In [5]:
import os

if not os.path.exists('/content/local_dataset/train/scratch'):
    print('로컬 디스크로 복사 중... (5~10분 소요)')
    !cp /content/drive/MyDrive/scratch_dataset.zip /content/
    !unzip -q /content/scratch_dataset.zip -d /content/local_dataset/
    print('로컬 캐시 완료!')
else:
    print('이미 로컬에 존재, 스킵')

# 데이터 확인
for split in ['train', 'val', 'test']:
    path = f'/content/local_dataset/{split}/scratch'
    if os.path.exists(path):
        cnt = sum(len(files) for _, _, files in os.walk(path))
        print(f'{split}/scratch: {cnt}개 파일')

로컬 디스크로 복사 중... (5~10분 소요)
로컬 캐시 완료!
train/scratch: 21504개 파일
val/scratch: 1920개 파일


## 6. Sanity Check
학습 전 모델과 loss가 정상 작동하는지 확인합니다.
patch_size=128로 테스트합니다.

In [6]:
import sys, torch
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')

from basicsr.archs.mambair_scratch_arch import MambaIRScratch
from basicsr.losses.scratch_losses import ScratchLoss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = MambaIRScratch(inp_channels=3, out_channels=3, dim=96, num_blocks=8).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,} ({params/1e6:.3f} M)')

# patch_size=128 테스트
dummy_lq = torch.rand(1, 3, 128, 128).to(device)
dummy_gt = torch.rand(1, 3, 128, 128).to(device)
with torch.no_grad():
    out = model(dummy_lq)
print(f'Output shape (128x128): {out.shape}')

loss_fn = ScratchLoss(w_char=1.0, w_edge=0.2, w_ssim=0.1).to(device)
loss = loss_fn(out, dummy_gt)
print(f'Loss: {loss.item():.4f}')
print('Sanity check 통과!')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Device: cuda
Parameters: 1,189,163 (1.189 M)
Output shape (128x128): torch.Size([1, 3, 128, 128])
Loss: 0.6918
Sanity check 통과!


## 7. 학습 YAML 생성
**수정 가능한 설정값:**
- `total_iter`: 학습 반복 횟수 (200000 권장)
- `resume_state`: 이어서 학습할 .state 파일 경로 (처음이면 `~` 유지)
- `gt_size`: 128 (V2 대비 2배, 더 넓은 context)
- `w_edge`: 0.2 (스크래치 경계 강조)
- `w_fft`: 0.1 (주파수 손실 추가)

In [7]:
import os

# ── 여기만 수정 ──────────────────────────────────────
task         = 'scratch'
difficulty   = 'hard'
width        = 96
num_blocks   = 8
total_iter   = 100000
# resume_state = '~'   # 처음 시작
resume_state = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim96-V3/training_states/96000.state'
# 이어서 하려면 아래 주석 해제 후 iter 번호 수정
# resume_state = '/content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim96-V2/training_states/74000.state'
exp_name     = f'MambaIRScratch-SIDL-{task}-dim{width}-V3'
# ─────────────────────────────────────────────────────

out_dir = '/content/drive/MyDrive/sidl_options/train'
os.makedirs(out_dir, exist_ok=True)

template = f'''name: {exp_name}
model_type: SRModel
scale: 1
num_gpu: 1
manual_seed: 42

path:
  root: /content/drive/MyDrive/MambaIR/experiments
  pretrain_network_g: ~
  strict_load_g: true
  resume_state: {resume_state}

datasets:
  train:
    name: sidl-{task}-train
    type: PairedImageDataset
    dataroot_gt: /content/local_dataset/train/{task}/target
    dataroot_lq: /content/local_dataset/train/{task}/input
    io_backend:
      type: disk
    gt_size: 96
    use_hflip: true
    use_rot: true
    use_shuffle: true
    num_worker_per_gpu: 4
    batch_size_per_gpu: 16
  val:
    name: sidl-{task}-{difficulty}-val
    type: PairedImageDataset
    dataroot_gt: /content/local_dataset/val/{task}/{difficulty}/target
    dataroot_lq: /content/local_dataset/val/{task}/{difficulty}/input
    io_backend:
      type: disk

network_g:
  type: MambaIRScratch
  inp_channels: 3
  out_channels: 3
  dim: {width}
  num_blocks: {num_blocks}
  d_state: 16
  d_conv: 4
  expand: 2
  img_range: 1.0

train:
  optim_g:
    type: Adam
    lr: !!float 2e-5
    weight_decay: !!float 0
    betas: [0.9, 0.999]
  scheduler:
    type: CosineAnnealingLR
    T_max: {total_iter}
    eta_min: !!float 1e-7
  total_iter: {total_iter}
  warmup_iter: 5000
  pixel_opt:
    type: ScratchLoss
    loss_weight: 1.0
    w_char: 1.0
    w_edge: 0.2
    w_ssim: 0.1
    w_fft: 0.0001
    eps: !!float 1e-3

val:
  val_freq: !!float 10000
  save_img: false
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 00
      test_y_channel: false

logger:
  print_freq: 100
  save_checkpoint_freq: !!float 2000
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~

dist_params:
  backend: nccl
  port: 29500
'''

out_path = os.path.join(out_dir, f'MambaIRScratch_{task}_v3.yml')
with open(out_path, 'w') as f:
    f.write(template)
print(f'YAML 생성 완료: {out_path}')
print(f'실험명: {exp_name}')
print(f'total_iter: {total_iter}, resume: {resume_state}')

YAML 생성 완료: /content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch_v3.yml
실험명: MambaIRScratch-SIDL-scratch-dim96-V3
total_iter: 100000, resume: /content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim96-V3/training_states/96000.state


In [8]:
path = '/content/drive/MyDrive/MambaIR/basicsr/models/sr_model.py'

with open(path, 'r') as f:
    content = f.read()

old = 'l_total.backward()\n        self.optimizer_g.step()'
new = '''l_total.backward()
        torch.nn.utils.clip_grad_norm_(self.net_g.parameters(), max_norm=1.0)
        self.optimizer_g.step()'''

if old in content and 'clip_grad_norm_' not in content:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('grad clip 패치 완료!')
else:
    print('이미 적용됨 or 코드 패턴 다름')

with open(path, 'r') as f:
    for line in f:
        if 'backward' in line or 'clip_grad' in line or 'optimizer_g.step' in line:
            print(line.rstrip())

이미 적용됨 or 코드 패턴 다름
        l_total.backward()
        torch.nn.utils.clip_grad_norm_(self.net_g.parameters(), max_norm=1.0)
        self.optimizer_g.step()


## 8. 학습 실행
200K iter 기준 A100에서 약 6~8시간 소요 (patch_size 128로 증가).
세션이 끊겨도 체크포인트에서 resume 가능합니다.

In [9]:
import os
os.chdir('/content/drive/MyDrive/MambaIR')

!torchrun \
  --standalone \
  --nnodes=1 \
  --nproc_per_node=1 \
  --master_port=4321 \
  basicsr/train.py \
    -opt /content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch_v3.yml \
    --launcher pytorch

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Set pretrain_network_g to /content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim96-V3/models/net_g_96000.pth
2026-06-04 04:43:53,036 INFO: 
                ____                _       _____  ____
               / __ ) ____ _ _____ (_)_____/ ___/ / __ \
              / __  |/ __ `// ___// // ___/\__ \ / /_/ /
             / /_/ // /_/ /(__  )/ // /__ ___/ // _, _/
            /_____/ \__,_//____//_/ \___//____//_/ |_|
     ______                   __   __                 __      __
    / ____/____   ____   ____/ /  / /   __  __ _____ / /__   / /
   / / __ / __ \ / __ \ / __  /  / /   / / / // ___// //_/  / /
  / /_/ // /_/ // /_/ // /_/ /  / /___/ /_/ // /__ / /<    /_/
  \____/ \____/ \____/ \___

## 9. 추론 - TTA (Test Time Augmentation)
8방향 flip/rotate 앙상블로 **+0.2~0.5dB** 성능 향상.
학습 완료 후 `ckpt_iter`를 실제 번호로 수정하세요.

In [12]:
import os, torch, glob, sys
import numpy as np
from PIL import Image
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')
from basicsr.archs.mambair_scratch_arch import MambaIRScratch

def tta_inference(model, lq_tensor):
    """8방향 TTA 앙상블 (flip x rotate)"""
    preds = []
    for flip in [False, True]:
        for rot in [0, 1, 2, 3]:
            x = lq_tensor.clone()
            if flip:
                x = torch.flip(x, dims=[-1])
            x = torch.rot90(x, k=rot, dims=[-2, -1])
            with torch.no_grad():
                pred = model(x)
            pred = torch.rot90(pred, k=-rot, dims=[-2, -1])
            if flip:
                pred = torch.flip(pred, dims=[-1])
            preds.append(pred)
    return torch.stack(preds).mean(0)

# ── 설정 ──
ckpt_iter = 98000
exp_name  = 'MambaIRScratch-SIDL-scratch-dim96-V3'
ckpt_path = f'/content/drive/MyDrive/MambaIR/experiments/{exp_name}/models/net_g_{ckpt_iter}.pth'
task      = 'scratch'
# ──────────

device = torch.device('cuda')
model  = MambaIRScratch(inp_channels=3, out_channels=3, dim=96, num_blocks=8).to(device)
state  = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state['params'], strict=True)
model.eval()
print(f'체크포인트 로드: {ckpt_path}')

for difficulty in ['easy', 'medium', 'hard']:
    lq_dir  = f'/content/drive/MyDrive/test/{task}/{difficulty}/input'
    out_dir = f'/content/drive/MyDrive/MambaIR/results/{exp_name}-{difficulty}-test/visualization/sidl-{task}-{difficulty}-test'
    os.makedirs(out_dir, exist_ok=True)
    lq_files = sorted(glob.glob(os.path.join(lq_dir, '*.png')))
    total = len(lq_files)
    print(f'\n[{difficulty}] {total}장 TTA 추론 시작...')
    for i, lq_path in enumerate(lq_files):
        img = np.array(Image.open(lq_path).convert('RGB')).astype(np.float32) / 255.0
        lq_t = torch.from_numpy(img.transpose(2,0,1)).unsqueeze(0).to(device)
        pred = tta_inference(model, lq_t)
        pred = pred.squeeze(0).permute(1,2,0).clamp(0,1).cpu().numpy()
        Image.fromarray((pred*255).astype(np.uint8)).save(
            os.path.join(out_dir, os.path.basename(lq_path))
        )
        if (i+1) % 50 == 0 or (i+1) == total:
            print(f'  [{difficulty}] {i+1}/{total}장 완료')
    print(f'[{difficulty}] 완료 -> {out_dir}')

print('\nTTA 추론 전체 완료!')

체크포인트 로드: /content/drive/MyDrive/MambaIR/experiments/MambaIRScratch-SIDL-scratch-dim96-V3/models/net_g_98000.pth

[easy] 672장 TTA 추론 시작...
  [easy] 50/672장 완료
  [easy] 100/672장 완료
  [easy] 150/672장 완료
  [easy] 200/672장 완료
  [easy] 250/672장 완료
  [easy] 300/672장 완료
  [easy] 350/672장 완료
  [easy] 400/672장 완료
  [easy] 450/672장 완료
  [easy] 500/672장 완료
  [easy] 550/672장 완료
  [easy] 600/672장 완료
  [easy] 650/672장 완료
  [easy] 672/672장 완료
[easy] 완료 -> /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-dim96-V3-easy-test/visualization/sidl-scratch-easy-test

[medium] 480장 TTA 추론 시작...
  [medium] 50/480장 완료
  [medium] 100/480장 완료
  [medium] 150/480장 완료
  [medium] 200/480장 완료
  [medium] 250/480장 완료
  [medium] 300/480장 완료
  [medium] 350/480장 완료
  [medium] 400/480장 완료
  [medium] 450/480장 완료
  [medium] 480/480장 완료
[medium] 완료 -> /content/drive/MyDrive/MambaIR/results/MambaIRScratch-SIDL-scratch-dim96-V3-medium-test/visualization/sidl-scratch-medium-test

[hard] 768장 TTA 추론 시작...
  [hard

## 10. 제출 파일 생성 (.tar)
추론 결과를 SIDL 제출 형식에 맞게 압축합니다.
`sidl_submission_v3.tar`가 Drive에 저장됩니다.

In [13]:
import os, shutil, glob, tarfile

task             = 'scratch'
exp_name         = 'MambaIRScratch-SIDL-scratch-dim96-V3'
difficulty_levels = ['easy', 'medium', 'hard']

source_dirs = {
    diff: f'/content/drive/MyDrive/MambaIR/results/{exp_name}-{diff}-test/visualization/sidl-{task}-{diff}-test'
    for diff in difficulty_levels
}

temp_dir = '/content/temp_submission_v3'
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
task_dir = os.path.join(temp_dir, task)
os.makedirs(task_dir)

for diff in difficulty_levels:
    input_dir = os.path.join(task_dir, diff, 'input')
    os.makedirs(input_dir, exist_ok=True)
    png_files = glob.glob(os.path.join(source_dirs[diff], '*.png'))
    print(f'{diff}: {len(png_files)}개 파일')
    for f_path in png_files:
        shutil.copy(f_path, os.path.join(input_dir, os.path.basename(f_path)))

tar_path = '/content/drive/MyDrive/sidl_submission_v3.tar'
with tarfile.open(tar_path, 'w') as tar:
    tar.add(task_dir, arcname=task)

print(f'\n제출 파일 생성 완료: {tar_path}')
shutil.rmtree(temp_dir)

easy: 672개 파일
medium: 480개 파일
hard: 768개 파일

제출 파일 생성 완료: /content/drive/MyDrive/sidl_submission_v3.tar


## 기타사항


In [ ]:
import os

# test 폴더 전체 구조 확인
test_root = '/content/local_dataset/test'
if os.path.exists(test_root):
    for root, dirs, files in os.walk(test_root):
        depth = root.replace(test_root, '').count(os.sep)
        if depth < 4:  # 너무 깊이 들어가지 않게
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/')
else:
    print('test 폴더 자체가 없음!')

    # zip 안에 뭐가 있는지 확인
    !unzip -l /content/scratch_dataset.zip | grep test | head -20

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/scratch_dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    all_names = z.namelist()

    # 전체 최상위 구조만
    print('[ zip 내부 구조 ]')
    seen = set()
    for name in all_names:
        parts = name.split('/')
        top3 = '/'.join(parts[:3])
        if top3 not in seen:
            seen.add(top3)
            print(top3)

In [ ]:
import os

# Drive에서 test 폴더 직접 찾기
print('Drive에서 test 폴더 찾는 중...')
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for d in dirs:
        if 'test' in d.lower():
            full_path = os.path.join(root, d)
            print(full_path)

In [ ]:
import os

# scratch 관련만 빠르게 찾기
!find /content/drive/MyDrive -maxdepth 3 -name "*test*" -o -name "*scratch*" 2>/dev/null

/content/drive/MyDrive/Colab Notebooks/CV_test.ipynb
/content/drive/MyDrive/Colab Notebooks/SIDL_NAFNet_scratch.ipynb
/content/drive/MyDrive/Colab Notebooks/SIDL_NAFNet_scratch_1.ipynb
/content/drive/MyDrive/Colab Notebooks/SIDL_NAFNet_scratch_3.ipynb
/content/drive/MyDrive/Colab Notebooks/SIDL_NAFNet_scratch_1147.ipynb
/content/drive/MyDrive/다운받은것들/학교/sat-practice-test-1.pdf
/content/drive/MyDrive/network/scratch-simulator.cc
/content/drive/MyDrive/SIDL_Research/data/test
/content/drive/MyDrive/SIDL_Research/configs/test
/content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch.yml
/content/drive/MyDrive/sidl_options/train/MambaIRScratch_scratch_v3.yml
/content/drive/MyDrive/sidl_options/test
/content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_easy.yml
/content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_hard.yml
/content/drive/MyDrive/sidl_options/test/MambaIRScratch_scratch_medium.yml
/content/drive/MyDrive/test
/content/drive/MyDrive/NAFN

In [ ]:
import os

# test 데이터 구조 확인
test_path = '/content/drive/MyDrive/SIDL_Research/data/test'
for root, dirs, files in os.walk(test_path):
    depth = root.replace(test_path, '').count(os.sep)
    if depth < 4:
        indent = '  ' * depth
        cnt = len(files)
        print(f'{indent}{os.path.basename(root)}/  ({cnt}개)')

test/  (0개)
  scratch/  (0개)
    easy/  (0개)
      input/  (0개)
    medium/  (0개)
      input/  (0개)
    hard/  (0개)
      input/  (0개)


In [ ]:
# 실제 Drive 경로인지 확인
import os

path = '/content/drive/MyDrive/SIDL_Research/data/test/scratch/easy/input'
print('실제 경로 존재:', os.path.exists(path))

# Drive 마운트 상태 확인
!ls /content/drive/MyDrive/SIDL_Research/data/test/scratch/easy/input | head -5

실제 경로 존재: True


In [ ]:
import os

# 실제 파일 있는지 확인
for diff in ['easy', 'medium', 'hard']:
    path = f'/content/drive/MyDrive/test/scratch/{diff}/input'
    if os.path.exists(path):
        cnt = len([f for f in os.listdir(path) if f.endswith('.png')])
        print(f'{diff}/input: {cnt}개')
    else:
        print(f'없음: {path}')

easy/input: 672개
medium/input: 480개
hard/input: 768개


In [15]:
!pip install thop -q

In [16]:
import torch, sys
sys.path.insert(0, '/content/drive/MyDrive/MambaIR')
from basicsr.archs.mambair_scratch_arch import MambaIRScratch
from thop import profile

device = torch.device('cuda')
model = MambaIRScratch(inp_channels=3, out_channels=3, dim=96, num_blocks=8).to(device)

# 입력 크기 (1장, 3채널, 256x256 기준)
dummy = torch.randn(1, 3, 256, 256).to(device)
macs, params = profile(model, inputs=(dummy,), verbose=False)

print(f'GMACs: {macs/1e9:.3f}')
print(f'Parameters: {params/1e6:.3f} M')

GMACs: 41.366
Parameters: 0.641 M
